# 03 — Evaluate a trained model on the test set (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sissikhaled/medical-text-simplifier/blob/main/notebooks/03_evaluate_colab.ipynb)

Runs `src/evaluate_model.py` on a Colab GPU: generates a simplification for each of the 480 test abstracts with a model trained by notebook 02, then computes all metrics. It uses the `best/` model and the run files that notebook 02 saved on Google Drive.

Progress is saved to Drive after every 20 examples. If the session dies, reconnect and **Run all**: it continues where it stopped.

The same evaluation also runs on a Mac (slower, but no GPU quota):
```
python -m src.evaluate_model --run <run_name> --model-dir models/<run_name>
```

## 1. Settings

In [ ]:
# Which trained run to evaluate (its best/ folder must be on Drive):
#   flan-t5-base_lr3e-4_src1024   main run
#   flan-t5-base_lr1e-4_src1024   variation (a)
#   flan-t5-base_lr3e-4_src512    variation (b)
RUN = "flan-t5-base_lr3e-4_src1024"

# "beam4" (default decoding) or "greedy" (decoding comparison, main run only)
DECODING = "beam4"

REPO_URL = "https://github.com/sissikhaled/medical-text-simplifier.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/medical-text-simplifier"

## 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
MODEL_DIR = f"{DRIVE_ROOT}/checkpoints/{RUN}/best"
RUNS_DIR = f"{DRIVE_ROOT}/runs"
assert os.path.isdir(MODEL_DIR), f"No model at {MODEL_DIR}: train this run with notebook 02 first"
assert os.path.exists(f"{RUNS_DIR}/{RUN}/config.yaml"), f"No run files at {RUNS_DIR}/{RUN}"
print("Model:", MODEL_DIR)

## 4. Code and packages

In [ ]:
%cd /content
if os.path.isdir("medical-text-simplifier/.git"):
    !git -C medical-text-simplifier fetch -q origin {BRANCH} && git -C medical-text-simplifier reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO_URL}
%cd /content/medical-text-simplifier
!git log --oneline -1
!grep -v -E "^(torch|pandas|matplotlib|jupyter|gradio)==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 5. Generate and score
Resumes automatically from the saved progress on Drive.

In [ ]:
!python -m src.evaluate_model --run {RUN} --model-dir "{MODEL_DIR}" --decoding {DECODING} --runs-dir "{RUNS_DIR}" --batch-size 8

## 6. Download the results
Downloads `<run>_test_<decoding>.zip` (predictions + metrics). Send it back so it can be committed.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN}_test_{DECODING}", "zip", f"{RUNS_DIR}/{RUN}", f"test_{DECODING}")
files.download(zip_path)